In [2]:
# Imports
import pyfabricops as pf
from dotenv import load_dotenv
import os

In [ ]:
# Parameters
root_path                   = os.path.abspath(os.path.join(os.getcwd(), '..'))
project_name                = 'healthy_analytics'
capacity                    = 'TrialBrazilSouth'

# Suffixes for workspaces based on branch name
branches = [      
    {"branch": "develop", "suffix": "DEV"},
    {"branch": "main",    "suffix": "PRD"}, 
]

# Users to add to workspaces and connections
roles = [
    {
        "user_uuid": "8ff12cf8-7085-457e-a5b2-566c96a74076",
        "user_type": "User",
    },
    {
        "user_uuid": "dd84b6c5-eb46-42e5-9a9f-2e610990b39a",
        "user_type": "ServicePrincipal",
    },
]

In [ ]:
capacity                    = 'TrialBrazilSouth'

# Suffixes for workspaces based on branch name
branches = [      
    {"branch": "develop", "suffix": "DEV"},
    {"branch": "main",    "suffix": "PRD"}, 
]

# Users to add to workspaces and connections
roles = [
    {
        "user_uuid": "8ff12cf8-7085-457e-a5b2-566c96a74076",
        "user_type": "User",
    },
    {
        "user_uuid": "dd84b6c5-eb46-42e5-9a9f-2e610990b39a",
        "user_type": "ServicePrincipal",
    },
]

In [4]:
# Authentication and logging
load_dotenv()
pf.set_auth_provider('env')
pf.setup_logging(
    level='info', 
    format_style='detailed',
)
logger = pf.get_logger(__name__)

In [5]:
# Create workspaces
workspaces = []
capacity_id = pf.resolve_capacity(capacity)

for branch in branches:
    branch_suffix = branch['suffix']

    workspace_name = f'{project_name}-{branch_suffix}'
    
    workspace_created = pf.create_workspace(
        workspace_name,
        capacity=capacity_id,
        description='End-to-end project using PySUS to ingest data from DATASUS to Microsoft Fabric.',
        df=False
    )

    # Append workspace info to the list
    if workspace_created:
        workspaces.append(workspace_created)
        workspace_id = workspace_created['id']
        logger.success(f'Workspace "{workspace_name}" created with ID: {workspace_id}')
    else:
        workspace_retrieved = pf.get_workspace(workspace_name, df=False)
        workspaces.append(workspace_retrieved)
        workspace_id = workspace_retrieved['id']
        logger.warning(f'Workspace "{workspace_name}" already exists with ID: {workspace_id}')

    # Assing roles to workspace
    for role in roles:
        pf.add_workspace_role_assignment(
            workspace_name,
            role['user_uuid'],
            role['user_type'],
            role='Admin',
            df=False
        )

display(pf.json_to_df(workspaces))

✓ 06:27:52 | pyfabricops.main     | SUCCESS  | Workspace "healthy_analytics-DEV" created with ID: 3c1f55c6-af24-4982-9f5e-840754f4b3ff
△ 06:27:54 | pyfabricops.api.api  | WARNING  | 409: {"requestId":"586f630d-3711-46a9-a54b-7e41015d7b54","errorCode":"PrincipalAlreadyHasWorkspaceRolePermissions","moreDetails":[{"relatedResource":{"resourceId":"dd84b6c5-eb46-42e5-9a9f-2e610990b39a","resourceType":"Principal"}}],"message":"The provided principal already has a role assigned in the workspace","relatedResource":{"resourceId":"3c1f55c6-af24-4982-9f5e-840754f4b3ff","resourceType":"Workspace"}}.
✓ 06:28:07 | pyfabricops.main     | SUCCESS  | Workspace "healthy_analytics-PRD" created with ID: 2ce2ac92-57b3-4ab1-98ce-56686f0cae4d
△ 06:28:10 | pyfabricops.api.api  | WARNING  | 409: {"requestId":"23020348-b1d3-4e3d-a953-1f1938a5a110","errorCode":"PrincipalAlreadyHasWorkspaceRolePermissions","moreDetails":[{"relatedResource":{"resourceId":"dd84b6c5-eb46-42e5-9a9f-2e610990b39a","resourceType":"Princ

,id,displayName,description,type,capacityId
0,3c1f55c6-af24-4982-9f5e-840754f4b3ff,healthy_analytics-DEV,End-to-end project using PySUS to ingest data ...,Workspace,66363f83-af75-4870-9f8a-0432cd1ca188
1,2ce2ac92-57b3-4ab1-98ce-56686f0cae4d,healthy_analytics-PRD,End-to-end project using PySUS to ingest data ...,Workspace,66363f83-af75-4870-9f8a-0432cd1ca188


In [6]:
# Check capacity assign to workspaces
import time

MAX_RETRIES = 10
RETRY_DELAY = 10  # seconds
logger.info('Checking capacity assignment to workspaces...')

for branch in branches:
    suffix = branch['suffix']

    for attempt in range(1, MAX_RETRIES + 1):
        logger.info(f'Attempt {attempt}/{MAX_RETRIES}')
        workspace = pf.get_workspace(f'{project_name}-{suffix}', df=False)

        if workspace['capacityAssignmentProgress'] == 'Completed':
            logger.success(f'Capacity assigned successfully to workspace {workspace["displayName"]}.')
            break
        else:
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
            else:
                logger.error(f'Capacity assignment failed after {MAX_RETRIES} attempts.')
                raise TimeoutError(f'Capacity assignment did not complete within {MAX_RETRIES * RETRY_DELAY} seconds.')

i 06:28:10 | pyfabricops.main     | INFO     | Checking capacity assignment to workspaces...
i 06:28:10 | pyfabricops.main     | INFO     | Attempt 1/10
✓ 06:28:11 | pyfabricops.main     | SUCCESS  | Capacity assigned successfully to workspace healthy_analytics-DEV.
i 06:28:11 | pyfabricops.main     | INFO     | Attempt 1/10
✓ 06:28:13 | pyfabricops.main     | SUCCESS  | Capacity assigned successfully to workspace healthy_analytics-PRD.


In [7]:
# Create Folders, Lakehouses and Environments in each workspace
for workspace in workspaces:

    folders = ['data', 'utils', 'notebooks', 'pipelines', 'reporting']
    for folder in folders:
        pf.create_folder(
            workspace=workspace['id'],
            display_name=folder,
            df=False,
        )
    
    pf.create_lakehouse(
        workspace=workspace['id'],
        display_name=f'lh_{project_name}',
        folder='data',
        enable_schemas=True,
        df=False,
    )

    environment_created = pf.create_environment(
        workspace=workspace['id'],
        display_name='env_default',
        folder='utils',
        df=False,
    )

    # If environment creation fails (e.g. because it already exists), retrieve the existing environment
    if environment_created is None:
        environment_created = pf.get_environment(
            workspace=workspace['id'],
            environment='env_default',
            df=False,
        )

    pf.update_environment_spark_compute(
        workspace=workspace['id'],
        environment='env_default',
        spark_properties=[
            {"key": "spark.sql.caseSensitive", "value": "true"}
        ],runtime_version='1.3'
    )

    pf.publish_environment(
        workspace=workspace['id'],
        environment=environment_created['id'],
        df=False,
    )

    pf.update_workspace_spark_settings(
        workspace=workspace['id'],
        automatic_log_enabled=True,
        high_concurrency_notebook_interactive_run_enabled=True,
        high_concurrency_notebook_pipeline_run_enabled=True,
        environment_name='env_default',
        environment_runtime_version='1.3',
        job_session_timeout_in_minutes=30,
        df=False,
    )

    logger.success(f'Folders, Lakehouses and Environments were created successfully in Workspace {workspace["displayName"]}')

✓ 06:28:26 | pyfabricops.main     | SUCCESS  | Folders, Lakehouses and Environments were created successfully in Workspace healthy_analytics-DEV
✓ 06:28:40 | pyfabricops.main     | SUCCESS  | Folders, Lakehouses and Environments were created successfully in Workspace healthy_analytics-PRD


In [8]:
# Create Variable Library Files for both workspaces
pf.delete_path('../tmp')

for b in branches:
    suffix = b['suffix']
    branch = b['branch']

    workspace = pf.get_workspace(f'{project_name}-{suffix}', df=False)

    lakehouse = pf.get_lakehouse(workspace=workspace['id'], lakehouse=f'lh_{project_name}', df=False)
    lakehouse_id = lakehouse['id']
    lakehouse_sql_endpoint_connection_string = lakehouse['properties']['sqlEndpointProperties']['connectionString']
    lakehouse_sql_endpoint_id = lakehouse['properties']['sqlEndpointProperties']['id']

    value_set = {
      "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/variableLibrary/definition/valueSet/1.0.0/schema.json",
      "name": branch,
      "variableOverrides": [
        {
          "name": "workspace_name",
          "value": workspace["displayName"]
        },
        {
          "name": "workspace_id",
          "value": workspace["id"]
        },
        {
          "name": "lakehouse_id",
          "value": lakehouse_id
        },
        {
          "name": "lakehouse_sql_endpoint_connection_string",
          "value": lakehouse_sql_endpoint_connection_string
        },
        {
          "name": "lakehouse_sql_endpoint_id",
          "value": lakehouse_sql_endpoint_id
        }
      ]
    }

    pf.write_json(
        value_set,
        f"../tmp/vl_variables.VariableLibrary/valueSets/{branch}.json"
    )

pf.write_json(
    {
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
        "metadata": {
            "type": "VariableLibrary",
            "displayName": "vl_variables",
            "description": "",
        },
        "config": {
            "version": "2.0",
            "logicalId": "00000000-0000-0000-0000-000000000000"
        }
    },
    "../tmp/vl_variables.VariableLibrary/.platform"
)

pf.write_json(
    {
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/variableLibrary/definition/settings/1.0.0/schema.json",
        "valueSetsOrder": [
            "develop",
            "main"
        ]
    },
    "../tmp/vl_variables.VariableLibrary/settings.json"
)

pf.write_json(
    {
      "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/variableLibrary/definition/variables/1.0.0/schema.json",
      "variables": [
        {
          "name": "workspace_id",
          "note": "",
          "type": "String",
          "value": ""
        },
        {
          "name": "workspace_name",
          "note": "",
          "type": "String",
          "value": ""
        },
        {
          "name": "lakehouse_id",
          "note": "",
          "type": "String",
          "value": ""
        },
        {
          "name": "lakehouse_sql_endpoint_connection_string",
          "note": "",
          "type": "String",
          "value": ""
        },
        {
          "name": "lakehouse_sql_endpoint_id",
          "note": "",
          "type": "String",
          "value": ""
        },
      ]
    },
    "../tmp/vl_variables.VariableLibrary/variables.json"
)

logger.success(f'Created the temporary Variable Library template in ../tmp/vl_variables.VariableLibrary')

✓ 06:28:46 | pyfabricops.main     | SUCCESS  | Created the temporary Variable Library template in ../tmp/vl_variables.VariableLibrary


In [9]:
# Create Variable Library in both workspaces
for b in branches:
    suffix = b['suffix']
    pf.create_variable_library(
        workspace=f'{project_name}-{suffix}',
        display_name='vl_variables',
        item_definition=pf.pack_item_definition(path="../tmp/vl_variables.VariableLibrary"),
        folder='utils',
        df=False,
    )
    logger.success(f'Variable Library vl_variables created in workspace {project_name}-{suffix}')

✓ 06:28:56 | pyfabricops.main     | SUCCESS  | Variable Library vl_variables created in workspace healthy_analytics-DEV
✓ 06:29:05 | pyfabricops.main     | SUCCESS  | Variable Library vl_variables created in workspace healthy_analytics-PRD


In [10]:
# Create GitHub connection and connect to DEV workspace
connection_name = f'conn_github_{project_name}'
workspace_id = pf.resolve_workspace(f'{project_name}-DEV')
repository_name = 'end-to-end-healthy-analytics'
repository_url = f'https://github.com/jaircampelo/{repository_name}'

github_connection = pf.create_github_source_control_connection(
    display_name=connection_name,
    repository=repository_url,
    github_token=os.getenv('GH_TOKEN'),
    df=False,
)

# If connection fail (e.g. beacause it already exists), retrieve the existing connection
if github_connection is not None:
    display(github_connection)

connection_id = pf.resolve_connection(connection_name)

# Assing owners and users to connection
for role in roles:
    if role['user_type'] == 'Group':
        pf.add_connection_role_assignment(
            connection_name,
            role['user_uuid'],
            role['user_type'],
            role='User',
            df=False,
        )
    else:
        pf.add_connection_role_assignment(
        connection_name,
        role['user_uuid'],
        role['user_type'],
        role='Owner',
        df=False,
    )
        
# Connect Github repository to DEV workspace
pf.github_connect(
    workspace=workspace_id,
    connection=connection_id,
    owner_name='jaircampelo',
    repository_name=repository_name,
    branch_name='develop',
    directory_name='src',
)

# If the repository was already connected, update the connection settings to ensure they are correct
pf.update_my_git_connection(
    workspace=workspace_id,
    request_body_type='UpdateGitCredentialsToConfiguredConnectionRequest',
    connection_id=connection_id,
)

# Innitialize Git repository in workspace and make initial commit
pf.git_init(workspace=workspace_id, initialize_strategy='PreferWorkspace', df=False)
pf.commit_to_git(workspace=workspace_id, mode='All', comment='Project Start', df=False)

△ 06:29:09 | pyfabricops.api.api  | WARNING  | 409: {"requestId":"0430f144-2088-4b6b-ba09-b81ca4ab11d0","errorCode":"DuplicateConnectionName","message":"The connection DisplayName input is already being used by another connection"}.
△ 06:29:12 | pyfabricops.api.api  | WARNING  | 409: {"requestId":"6e10c99d-fc20-4093-9f9e-8b010b2e29d4","errorCode":"ConnectionRoleAssignmentAlreadyExists","moreDetails":[{"relatedResource":{"resourceId":"8ff12cf8-7085-457e-a5b2-566c96a74076","resourceType":"Principal"}}],"message":"Connection role assignment already exists","relatedResource":{"resourceId":"bb20ed70-3dc6-4f80-b6df-765a39bcad54","resourceType":"Connection"}}.
△ 06:29:14 | pyfabricops.api.api  | WARNING  | 409: {"requestId":"aade2350-e6c8-4160-a815-9c1b54821729","errorCode":"ConnectionRoleAssignmentAlreadyExists","moreDetails":[{"relatedResource":{"resourceId":"dd84b6c5-eb46-42e5-9a9f-2e610990b39a","resourceType":"Principal"}}],"message":"Connection role assignment already exists","relatedRes